# Debug Mode

Start and inspect the Tango control stack in database mode before running acquisition notebooks.

Make sure you are on the VPN, the AutoScript server is running, and the Tango servers have been started with:

```bash
uv run scripts/3_run_servers.py
```


### Alternative local launch


In [ ]:
import sys
from pathlib import Path
import time
import json

# For resolving ModuleNotFoundErrors
notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent.resolve()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import subprocess
import numpy as np
import matplotlib.pyplot as plt
from asyncroscopy.ThermoMicroscope import ThermoMicroscope
from tango.test_context import MultiDeviceTestContext
from asyncroscopy.detectors.EDS import EDS
from asyncroscopy.hardware.SCAN import SCAN

import tango

# ── Kill anything already on port 11000 ──────────────────────────────────────
print("Clearing old processes...")
subprocess.run("kill -9 $(lsof -t -i:11000) 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'STAGE stage' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'SCAN scan' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'ThermoMicroscope microscope_instance' 2>/dev/null || true", shell=True)
time.sleep(2)


devices_info = [
    {
        "class": SCAN,
        "devices": [
            {
                "name": "asyncroscopy/scan/default",
                "properties": {},
            }
        ],
    },

    {
        "class": EDS,
        "devices": [
            {
                "name": "asyncroscopy/eds/default",
                "properties": {},
            }
        ],
    },
    
    {
        "class": ThermoMicroscope,
        "devices": [
            {
                "name": "asyncroscopy/microscope/default",
                "properties": {
                    "eds_device_address": "asyncroscopy/eds/default",
                    "scan_device_address": "asyncroscopy/scan/default",
                },
            }
        ],
    },
]

ctx = MultiDeviceTestContext(devices_info, process=False)
ctx.start()

scan_proxy = tango.DeviceProxy("asyncroscopy/scan/default")
mic_proxy = tango.DeviceProxy("asyncroscopy/microscope/default")
eds_proxy = tango.DeviceProxy("asyncroscopy/eds/default")

print(f"Device state: {mic_proxy.state()}")


### Connect to devices


In [ ]:
import os

# Tango DB running on this
os.environ["TANGO_HOST"] = "localhost:9000"
# os.environ["TANGO_HOST"] = "10.46.217.241:9094"

In [ ]:
# list devices on DB
import tango

db = tango.Database()

devices = db.get_device_name("*", "*")

print("Devices registered in Tango DB:\n")

for d in devices:
    print(d)

### Interpret registered Tango devices


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tango


# --nodb mode — use full tango:// URL with port and #dbase=no suffix
microscope_proxy = tango.DeviceProxy("asyncroscopy/microscope/default")
microscope_proxy.set_timeout_millis(120_000)

scan_proxy = tango.DeviceProxy("asyncroscopy/scan/default")

In [ ]:
print('Microscope state:', microscope_proxy.state())


### Inspect device attributes and commands


In [ ]:
print('\n--- Microscope commands ---')
for cmd in microscope_proxy.get_command_list():
    print(f'  {cmd}')

### Configure HAADF acquisition


In [ ]:
scan_proxy.dwell_time 

In [ ]:
scan_proxy.dwell_time   = 10e-6   # 1 µs
scan_proxy.imsize  = 1024


print('dwell_time  :', scan_proxy.dwell_time)
print('image_width :', scan_proxy.imsize)


### Acquire a HAADF image


In [ ]:
# get_image returns DevEncoded = (json_metadata_str, raw_bytes)
json_meta, raw_bytes = microscope_proxy.get_scanned_image()

metadata  = dict(json.loads(json_meta))
image = np.frombuffer(raw_bytes, dtype=metadata['dtype']).reshape(metadata['shape'])

print('Metadata:', metadata)
print('Image shape:', image.shape)
print('Image dtype:', image.dtype)

### Display the image


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray', interpolation='none')
ax.set_title(f"HAADF — dwell {metadata['dwell_time']*1e6:.1f} µs")
ax.axis('off')
plt.tight_layout()
plt.show()

### Build a sidpy dataset


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

import sidpy
print('sidpy version: ', sidpy.__version__)

In [ ]:
dataset = sidpy.Dataset.from_array(image , name='random')

# set dimesnions
dataset.set_dimension(0, sidpy.Dimension(np.arange(image.shape[0])*.02, 'x'))
dataset.set_dimension(1, sidpy.Dimension(np.arange(image.shape[0])*.02, 'y'))


# set the dataset level plotting metadata
dataset.data_type = 'image'
dataset.units = 'counts'
dataset.quantity = 'intensity'
dataset.title = 'random'

# handle one dimension of the data
dataset.set_dimension(0, sidpy.Dimension(np.arange(dataset.shape[0])*.02, 'x'))
dataset.x.dimension_type = 'spatial'
dataset.x.units = 'nm'
dataset.x.quantity = 'distance'

# handle another dimension of the data

dataset.set_dimension(1, sidpy.Dimension(np.arange(dataset.shape[1])*.02, 'y'))
dataset.y.dimension_type = 'spatial'
dataset.y.units = 'nm'
dataset.y.quantity = 'distance'


In [ ]:
dataset.metadata = metadata

In [ ]:
view = dataset.plot(scale_bar=True)

### Advanced multi-detector acquisition


In [ ]:
adv_acq_proxy = tango.DeviceProxy("tango://127.0.0.1:8890/asyncroscopy/advancedacquisition/default#dbase=no")


In [ ]:
adv_acq_proxy.state()

In [ ]:
for attr in adv_acq_proxy.get_attribute_list():
    print(f'  {attr}')



In [ ]:
adv_acq_proxy.dwell_time

In [ ]:
adv_acq_proxy.scan_region

In [ ]:
# One simultaneous acquisition
response = microscope_proxy.get_images(["HAADF", "HAADF"])
info = json.loads(response)


In [ ]:
import matplotlib.pyplot as plt

# Quick retrieval
images = []
for img_meta in info["images"]:
    meta_json, img_bytes = microscope_proxy.get_image_data_cached(img_meta["index"])
    img = np.frombuffer(img_bytes, dtype=img_meta["dtype"]).reshape(img_meta["shape"])
    images.append((img_meta["detector"], img))

# Plot all images
fig, axes = plt.subplots(1, len(images), figsize=(6*len(images), 5))

# Handle single image case
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, img) in zip(axes, images):
    im = ax.imshow(img, cmap='gray')
    ax.set_title(f"{detector_name.upper()} Detector")
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### Convert acquired images to sidpy datasets


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import sidpy

# Step 1: Acquire images simultaneously
response = microscope_proxy.get_images(["HAADF", "HAADF"])
info = json.loads(response)

# Step 2: Create sidpy datasets for each image
datasets = []

for img_meta in info["images"]:
    # Get the cached image
    meta_json, img_bytes = microscope_proxy.get_image_data(img_meta["index"])
    img = np.frombuffer(img_bytes, dtype=img_meta["dtype"]).reshape(img_meta["shape"])
    
    # Create sidpy dataset
    dataset = sidpy.Dataset.from_array(img, name=img_meta["detector"])
    
    # Set dimensions (assuming 0.02 nm/pixel - adjust as needed)
    dataset.set_dimension(0, sidpy.Dimension(np.arange(img.shape[0]) * 0.02, 'x'))
    dataset.set_dimension(1, sidpy.Dimension(np.arange(img.shape[1]) * 0.02, 'y'))
    
    # Set dataset metadata
    dataset.data_type = 'image'
    dataset.units = 'counts'
    dataset.quantity = 'intensity'
    dataset.title = img_meta["detector"].upper()
    
    # X dimension metadata
    dataset.x.dimension_type = 'spatial'
    dataset.x.units = 'nm'
    dataset.x.quantity = 'distance'
    
    # Y dimension metadata
    dataset.y.dimension_type = 'spatial'
    dataset.y.units = 'nm'
    dataset.y.quantity = 'distance'
    
    # Store acquisition metadata
    dataset.metadata = img_meta
    
    datasets.append(dataset)



In [ ]:
datasets